# Train Small Draft Model (43M) via Knowledge Distillation

**Teacher**: Qwen2.5-0.5B (494M params)  
**Student**: Small draft model (43M params, 4 layers, dim=256)  
**Goal**: Fast draft for ANE speculative decoding (>200 tok/s target)

**50k steps on H100 ≈ 40-60 min**

In [ ]:
!pip install -q torch transformers datasets safetensors sentencepiece

In [ ]:
import math, os, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from dataclasses import dataclass

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Define the small draft model

In [ ]:
SMALL_CONFIG = {
    "vocab_size": 151936,
    "hidden_size": 256,
    "num_attention_heads": 4,
    "num_key_value_heads": 2,
    "intermediate_size": 1024,
    "num_hidden_layers": 4,
    "rms_norm_eps": 1e-6,
    "rope_theta": 1000000.0,
    "max_seq_len": 128,
    "tie_word_embeddings": True,
}

@dataclass
class DraftModelConfig:
    vocab_size: int = 151936
    hidden_size: int = 256
    num_attention_heads: int = 4
    num_key_value_heads: int = 2
    intermediate_size: int = 1024
    num_hidden_layers: int = 4
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1000000.0
    max_seq_len: int = 128
    tie_word_embeddings: bool = True


def _precompute_rope(head_dim, max_seq_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()


def _rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def _apply_rope(q, k, cos, sin):
    return (q * cos) + (_rotate_half(q) * sin), (k * cos) + (_rotate_half(k) * sin)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        return self.weight * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps))


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh = cfg.num_attention_heads
        self.nkv = cfg.num_key_value_heads
        self.hd = cfg.hidden_size // cfg.num_attention_heads
        self.ng = self.nh // self.nkv
        self.q_proj = nn.Linear(cfg.hidden_size, self.nh * self.hd, bias=True)
        self.k_proj = nn.Linear(cfg.hidden_size, self.nkv * self.hd, bias=True)
        self.v_proj = nn.Linear(cfg.hidden_size, self.nkv * self.hd, bias=True)
        self.o_proj = nn.Linear(self.nh * self.hd, cfg.hidden_size, bias=False)

    def forward(self, x, cos, sin, mask):
        B, L, _ = x.shape
        q = self.q_proj(x).view(B, L, self.nh, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.nkv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.nkv, self.hd).transpose(1, 2)
        q, k = _apply_rope(q, k, cos, sin)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkv, self.ng, L, self.hd).reshape(B, self.nh, L, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkv, self.ng, L, self.hd).reshape(B, self.nh, L, self.hd)
        sc = 1.0 / math.sqrt(self.hd)
        a = torch.matmul(q, k.transpose(-2, -1)) * sc + mask
        a = F.softmax(a, dim=-1)
        return self.o_proj(torch.matmul(a, v).transpose(1, 2).contiguous().view(B, L, -1))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.up = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.down = nn.Linear(cfg.intermediate_size, cfg.hidden_size, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn = Attention(cfg)
        self.mlp = FeedForward(cfg)
        self.ln1 = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.ln2 = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.ln1(x), cos, sin, mask)
        return x + self.mlp(self.ln2(x))


class DraftModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.embed_tokens = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.num_hidden_layers)])
        self.norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        if cfg.tie_word_embeddings:
            self.lm_head.weight = self.embed_tokens.weight
        hd = cfg.hidden_size // cfg.num_attention_heads
        cos, sin = _precompute_rope(hd, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer('rope_cos', cos.unsqueeze(0).unsqueeze(0))
        self.register_buffer('rope_sin', sin.unsqueeze(0).unsqueeze(0))
        mask = torch.triu(torch.full((cfg.max_seq_len, cfg.max_seq_len), -1e9), diagonal=1)
        self.register_buffer('causal_mask', mask.unsqueeze(0).unsqueeze(0))

    def forward(self, input_ids):
        h = self.embed_tokens(input_ids)
        for layer in self.layers:
            h = layer(h, self.rope_cos, self.rope_sin, self.causal_mask)
        return self.lm_head(self.norm(h))

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())


cfg = DraftModelConfig(**SMALL_CONFIG)
student = DraftModel(cfg).to(device)
print(f"Student: {student.count_parameters():,} params")

## 2. Load teacher (Qwen2.5-0.5B)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

teacher = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B", torch_dtype=torch.float32
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
print(f"Teacher: {sum(p.numel() for p in teacher.parameters()):,} params")
print(f"Compression: {sum(p.numel() for p in teacher.parameters()) / student.count_parameters():.1f}x")

## 3. Prepare dataset (wikitext-103, 100M tokens)

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

SEQ_LEN = 128
all_ids = []
for text in ds["text"]:
    if text.strip():
        all_ids.extend(tokenizer.encode(text, add_special_tokens=False))
    if len(all_ids) > 20_000_000:  # cap at 20M tokens to save RAM
        break

n_chunks = len(all_ids) // SEQ_LEN
chunks = [torch.tensor(all_ids[i*SEQ_LEN:(i+1)*SEQ_LEN], dtype=torch.long) for i in range(n_chunks)]
loader = DataLoader(chunks, batch_size=32, shuffle=True, drop_last=True, num_workers=2)
print(f"Dataset: {n_chunks:,} chunks of {SEQ_LEN} tokens ({len(all_ids):,} total)")

## 4. Train (50k steps)

In [ ]:
STEPS = 50_000
LR = 5e-4
TEMP = 2.0
ALPHA = 0.7
BATCH = 32

optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)
warmup = 1000

def lr_fn(step):
    if step < warmup:
        return step / warmup
    return 0.5 * (1 + math.cos(math.pi * (step - warmup) / (STEPS - warmup)))

sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)

student.train()
data_iter = iter(loader)
losses = []
best_loss = float('inf')
t0 = time.time()

print(f"Training: {STEPS:,} steps | batch={BATCH} | lr={LR} | temp={TEMP}")
print(f"Dataset: {n_chunks:,} chunks | {len(all_ids):,} tokens")
print("-" * 65)

for step in range(1, STEPS + 1):
    try:
        ids = next(data_iter).to(device)
    except StopIteration:
        data_iter = iter(loader)
        ids = next(data_iter).to(device)

    inp = ids[:, :-1]
    labels = ids[:, 1:]
    B, L = inp.shape

    if L < SEQ_LEN:
        pad = torch.zeros(B, SEQ_LEN - L, dtype=torch.long, device=device)
        s_inp = torch.cat([inp, pad], dim=1)
    else:
        s_inp = inp[:, :SEQ_LEN]
        labels = labels[:, :SEQ_LEN]
        inp = inp[:, :SEQ_LEN]
        L = SEQ_LEN

    s_logits = student(s_inp)[:, :L, :]

    with torch.no_grad():
        t_logits = teacher(inp).logits

    V = min(s_logits.size(-1), t_logits.size(-1))
    sl, tl = s_logits[:,:,:V], t_logits[:,:,:V]

    kl = F.kl_div(
        F.log_softmax(sl / TEMP, dim=-1),
        F.softmax(tl / TEMP, dim=-1),
        reduction="batchmean"
    ) * TEMP**2

    ce = F.cross_entropy(sl.reshape(-1, V), labels.reshape(-1))
    loss = ALPHA * kl + (1 - ALPHA) * ce

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    optimizer.step()
    sched.step()

    losses.append(loss.item())

    if step % 500 == 0:
        avg = sum(losses[-500:]) / len(losses[-500:])
        elapsed = time.time() - t0
        eta = elapsed / step * (STEPS - step)
        if avg < best_loss:
            best_loss = avg
            marker = " *best*"
        else:
            marker = ""
        print(f"Step {step:6d}/{STEPS:,} | loss={avg:.4f}{marker} | "
              f"lr={sched.get_last_lr()[0]:.2e} | {elapsed/60:.1f}m | ETA {eta/60:.0f}m")

    # Checkpoint every 10k steps
    if step % 10_000 == 0:
        from safetensors.torch import save_file as sf
        os.makedirs("checkpoints", exist_ok=True)
        st = {k: v.cpu() for k, v in student.state_dict().items()
              if not k.startswith(("rope_", "causal_"))}
        sf(st, f"checkpoints/step_{step}.safetensors")
        print(f"  >> Checkpoint saved: checkpoints/step_{step}.safetensors")

print(f"\nDone! Total: {(time.time()-t0)/60:.1f} min | Best loss: {best_loss:.4f}")

## 5. Quick eval

In [ ]:
student.eval()

prompts = [
    "The meaning of life is",
    "Once upon a time",
    "The capital of France is",
]

for prompt in prompts:
    ids = tokenizer.encode(prompt)
    s_ctx = list(ids)
    t_ctx = list(ids)

    for _ in range(30):
        # Student
        padded = s_ctx[-SEQ_LEN:] + [0] * max(0, SEQ_LEN - len(s_ctx))
        inp = torch.tensor([padded[:SEQ_LEN]], device=device)
        with torch.no_grad():
            logits = student(inp)
        pos = min(len(s_ctx), SEQ_LEN) - 1
        s_ctx.append(logits[0, pos].argmax().item())

        # Teacher
        inp_t = torch.tensor([t_ctx], device=device)
        with torch.no_grad():
            logits_t = teacher(inp_t).logits
        t_ctx.append(logits_t[0, -1].argmax().item())

    s_out = tokenizer.decode(s_ctx[len(ids):])
    t_out = tokenizer.decode(t_ctx[len(ids):])
    match = sum(1 for a, b in zip(s_ctx[len(ids):], t_ctx[len(ids):]) if a == b)

    print(f'\nPrompt: "{prompt}"')
    print(f'Student: "{s_out}"')
    print(f'Teacher: "{t_out}"')
    print(f'Match: {match}/30 = {match/30:.0%}')

## 6. Save final weights

In [ ]:
from safetensors.torch import save_file

os.makedirs("draft_small_weights", exist_ok=True)

state = {k: v.cpu() for k, v in student.state_dict().items()
         if not k.startswith(("rope_", "causal_"))}
save_file(state, "draft_small_weights/draft_small.safetensors")

with open("draft_small_weights/config.json", "w") as f:
    json.dump(SMALL_CONFIG, f, indent=2)

sz = os.path.getsize("draft_small_weights/draft_small.safetensors")
print(f"Saved: draft_small.safetensors ({sz/1e6:.1f} MB)")
print(f"       config.json")

In [ ]:
try:
    from google.colab import files
    files.download("draft_small_weights/draft_small.safetensors")
    files.download("draft_small_weights/config.json")
except:
    print("Not on Colab - find files in draft_small_weights/")